# Roof line models: Point2Roof + BWFormer on 10 DFW homes (research test only)
Runtime > Change runtime type > **T4 GPU**, then Runtime > **Run all**. When asked, upload `linemodels-data.zip`
(from roof-command-sell/tour/audit/roof-accuracy/linemodels/). At the end the browser downloads `linemodels-results.zip`;
put it back in that same folder. BWFormer has no licence: research only, never shipped.

In [ ]:
!nvidia-smi -L
from google.colab import files
import zipfile, os
up = files.upload()                      # pick linemodels-data.zip
zipfile.ZipFile(next(iter(up))).extractall('/content')
rows = sorted(os.listdir('/content/data')); print(len(rows), 'homes:', rows)
os.makedirs('/content/results/p2r', exist_ok=True); os.makedirs('/content/results/bwf', exist_ok=True)

## 1. Point2Roof (MIT, checkpoint in the repo)

In [ ]:
%cd /content
!git clone -q https://github.com/Li-Li-Whu/Point2Roof.git
!pip -q install easydict pyyaml tqdm scikit-learn
%cd /content/Point2Roof/pc_util
!python setup.py install > /content/p2r_build.log 2>&1; tail -3 /content/p2r_build.log
%cd /content

In [ ]:
import sys, json, itertools, logging, functools, numpy as np, torch
np.float = float; np.int = int                          # the repo uses aliases removed from newer numpy
torch.load = functools.partial(torch.load, weights_only=False)
sys.path.insert(0, '/content/Point2Roof')
from utils import common_utils
from model.roofnet import RoofNet
from model import model_utils
cfg = common_utils.cfg_from_yaml_file('/content/Point2Roof/model_cfg.yaml')
net = RoofNet(cfg.MODEL).cuda(); model_utils.load_params(net, '/content/Point2Roof/checkpoint_epoch_90.pth', logging.getLogger())
net.eval(); net.use_edge = True
NP = cfg.DATA.NPOINT
for r in rows:
    pts = np.loadtxt(f'/content/data/{r}/points.xyz')[:, :3]
    rng = np.random.default_rng(0)
    idx = rng.integers(0, len(pts), NP) if len(pts) > NP else np.r_[np.arange(len(pts)), rng.integers(0, len(pts), NP - len(pts))]
    p = pts[idx]; lo, hi = p.min(), p.max()             # the repo's normalisation: one cube from the global min/max
    pn = ((p - lo) / (hi - lo)).astype(np.float32)
    batch = {'points': torch.from_numpy(pn)[None].cuda(), 'vectors': torch.full((1, 1, 3), -10.0).cuda(), 'edges': torch.full((1, 1, 2), -10.0).cuda(),
             'frame_id': [r], 'minMaxPt': [np.array([[lo] * 3, [hi] * 3], np.float32)], 'batch_size': 1}
    try:
        with torch.no_grad(): out = net(batch)
        kp = out['refined_keypoint'].cpu().numpy(); kb = out['keypoint'][:, 0].cpu().numpy(); V = kp[kb == 0] * (hi - lo) + lo
        sc = out['edge_score'].cpu().numpy(); pairs = out['pair_points'].cpu().numpy().astype(int)
        E = [[int(a), int(b)] for (a, b), s in zip(pairs, sc) if s > 0.5]
        json.dump({'vertices': V.tolist(), 'edges': E, 'n_points': int(len(pts))}, open(f'/content/results/p2r/{r}.json', 'w'))
        print(r, len(V), 'vertices', len(E), 'edges')
    except Exception as ex:
        print(r, 'FAILED', repr(ex)[:300]); json.dump({'error': repr(ex)[:500], 'vertices': [], 'edges': []}, open(f'/content/results/p2r/{r}.json', 'w'))

## 2. BWFormer (best effort; NO licence: research test only)
It works on a 256 x 256 height image of the roof points (the repo's proj.py idea). If a cell fails, the error is saved
and the Point2Roof results still download.

In [ ]:
%cd /content
!git clone -q https://github.com/3dv-casia/BWformer.git
!pip -q install einops gdown scikit-image shapely opencv-python
%cd /content/BWformer/models/ops
!sh make.sh > /content/bwf_build.log 2>&1; tail -3 /content/bwf_build.log
%cd /content/BWformer
!mkdir -p checkpoints && gdown -q 1qvwUUL8Zjpv33rb1F7Ew0DUUvv5tkqbs -O checkpoints/checkpoint.pth; ls -la checkpoints

In [ ]:
# height images in the layout infer.py reads (./building3d/b3d); the dataset class is printed so a mismatch is easy to see
import inspect, cv2
sys.path.insert(0, '/content/BWformer')
try:
    from datasets import test_outdoor_buildings as T
    print(inspect.getsource(T.testOutdoorBuildingDataset.__init__)[:1500])
except Exception as ex: print('dataset import failed', ex)
os.makedirs('/content/BWformer/building3d/b3d', exist_ok=True)
for r in rows:
    pts = np.loadtxt(f'/content/data/{r}/points.xyz')[:, :3]; lo = pts[:, :2].min(0); span = (pts[:, :2].max(0) - lo).max() + 1e-6
    ij = np.clip(((pts[:, :2] - lo) / span * 255).astype(int), 0, 255); img = np.zeros((256, 256), np.float32)
    for (i, j), z in zip(ij, pts[:, 2]): img[255 - j, i] = max(img[255 - j, i], z)
    img = (img / max(img.max(), 1e-6) * 255).astype(np.uint8); cv2.imwrite(f'/content/BWformer/building3d/b3d/{r}.jpg', img)
    json.dump({'lo': lo.tolist(), 'span': float(span)}, open(f'/content/BWformer/building3d/b3d/{r}.json', 'w'))
print('height images written')

In [ ]:
%cd /content/BWformer
!python infer.py --checkpoint_path ./checkpoints/checkpoint.pth --dataset outdoor --image_size 256 > /content/bwf_infer.log 2>&1; tail -20 /content/bwf_infer.log
# collect any .obj wireframes it wrote, mapped back to metres
import glob
for f in glob.glob('/content/BWformer/**/*.obj', recursive=True):
    r = os.path.splitext(os.path.basename(f))[0]
    if r not in rows: continue
    m = json.load(open(f'/content/BWformer/building3d/b3d/{r}.json')); V, E = [], []
    for line in open(f):
        t = line.split()
        if t and t[0] == 'v': x, y = float(t[1]), float(t[2]); V.append([m['lo'][0] + x / 255 * m['span'], m['lo'][1] + (255 - y) / 255 * m['span'], float(t[3]) if len(t) > 3 else 0.0])
        if t and t[0] == 'l': E.append([int(t[1]) - 1, int(t[2]) - 1])
    json.dump({'vertices': V, 'edges': E}, open(f'/content/results/bwf/{r}.json', 'w')); print('bwf', r, len(V), len(E))

## 3. Download the results

In [ ]:
!cp /content/*.log /content/results/ 2>/dev/null; cd /content && zip -qr linemodels-results.zip results
files.download('/content/linemodels-results.zip')